# Evaluating OpenCode Skills End-to-End with MLflow Scorers

This notebook demonstrates the end-to-end evaluation workflow for [OpenCode](https://github.com/opendatahub-io/opencode) skills on Red Hat OpenShift AI: run scorers against skill traces, find a gap, strengthen the skill, and re-evaluate.

The individual scorer behavior (what each failure mode looks like, how each scorer detects it) is covered in the [failure-mode notebooks](../failure-modes/). This notebook focuses on applying those scorers to OpenCode skill traces as a complete evaluation workflow — similar to the [LangGraph evaluation notebook](../langgraph-end-to-end-example/langgraph_agent_evaluation.ipynb), but for a coding agent.

**Workflow:**
1. Create traces that mirror real OpenCode skill executions observed on cluster
2. Run built-in + custom scorers (two-tier: deterministic → LLM judges)
3. Interpret results — discover a verification gap in `pr-summarizer`
4. Strengthen the skill and re-evaluate to confirm the fix

**Skills under evaluation:**

| Skill | What it does | Key tools |
|---|---|---|
| `python-file-review` | Reviews a Python file for quality issues, writes a markdown report | `tool_read`, `tool_bash` (ruff), `tool_write` |
| `pr-summarizer` | Summarizes a PR from a local git clone, writes a structured report | `tool_bash` (git), `tool_write`, `tool_read` |

## 1. Setup

Connect to MLflow tracking server and configure the judge model. Supports both a vLLM endpoint on OpenShift AI (`OPENAI_API_BASE`) and an OpenAI API key (`OPENAI_API_KEY`). Tier 2 LLM judge cells are skipped when neither is set.

In [ ]:
import os
import sys
from pathlib import Path

import mlflow
from dotenv import load_dotenv
from mlflow.entities import SpanType

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"

OPENCODE_DIR = next(
    (
        p.resolve()
        for p in [
            Path.cwd(),
            Path.cwd() / "opencode-end-to-end-example",
            Path.cwd()
            / "examples"
            / "agentic-evaluation"
            / "opencode-end-to-end-example",
        ]
        if (p / "golden_queries.json").exists()
    ),
    None,
)
if OPENCODE_DIR is None:
    raise FileNotFoundError("Cannot find golden_queries.json")

PROJECT_ROOT = OPENCODE_DIR.parent
sys.path.insert(0, str(OPENCODE_DIR))
os.chdir(str(PROJECT_ROOT))
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

_has_vllm = bool(os.environ.get("OPENAI_API_BASE"))
_has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
RUN_LLM_JUDGES = _has_vllm or _has_openai_key

if _has_vllm:
    JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai:/gpt-oss-20b")
    if not _has_openai_key:
        os.environ["OPENAI_API_KEY"] = "not-needed"
else:
    JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai:/gpt-4.1")

if os.environ.get("MLFLOW_TRACKING_INSECURE_TLS", "").lower() == "true":
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT = os.environ.get(
    "MLFLOW_EXPERIMENT_NAME", "opencode-scorer-evaluation"
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
client = mlflow.MlflowClient()

SKILL_TAGS = ["python-file-review", "pr-summarizer"]


def search_skill_traces():
    """MLflow trace search doesn't support OR/IN — query each skill separately."""
    traces = []
    for skill in SKILL_TAGS:
        traces.extend(
            mlflow.search_traces(
                locations=[EXPERIMENT.experiment_id],
                filter_string=f"tags.skill = '{skill}'",
                return_type="list",
            )
        )
    return traces


old_traces = search_skill_traces()
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old trace(s)")

print(f"Judge model: {JUDGE_MODEL}")
print(f"MLflow: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT}")
if _has_vllm:
    print(f"vLLM endpoint: {os.environ['OPENAI_API_BASE']}")
elif _has_openai_key:
    print("Using OpenAI API key for LLM judges")
else:
    print("Tier 2 LLM judges: SKIPPED (set OPENAI_API_BASE or OPENAI_API_KEY)")

## 2. Simulate OpenCode Skill Traces

In production, OpenCode's MLflow plugin captures traces automatically as the agent runs skills. Here we create synthetic traces that mirror real skill executions observed on an OpenShift AI cluster (OpenCode 1.18.3, `gpt-oss-20b` via vLLM).

Two traces, one per skill:
- **`python-file-review`** — reads a Python file, runs ruff, writes a report, reads it back to verify (clean run)
- **`pr-summarizer`** — fetches a PR, analyzes the diff, writes a summary, but **does not read it back** (matches the actual behavior observed on cluster)

In [ ]:
REVIEW_FILE = "/opt/app-root/workspace/input-files/data_pipeline.py"
REPORT_FILE = "/opt/app-root/workspace/reviews/data_pipeline-review.md"
PR_SUMMARY_FILE = "/opt/app-root/workspace/pr-summaries/pr-178-summary.md"

SAMPLE_PYTHON_SOURCE = (
    "import os\nimport json\n\ndef load_data(path):\n"
    "    with open(path) as f:\n        return json.load(f)\n\n"
    "def transform(data):\n    results = []\n"
    '    for item in data:\n        if item["status"] == "active":\n'
    "            results.append(item)\n    return results\n"
)
RUFF_OUTPUT_CLEAN = "All checks passed!"
REVIEW_REPORT = (
    "# Code Review: data_pipeline.py\n"
    "## Summary\nThe module handles data loading and filtering.\n"
    "## Issues\n### Medium\n- Unused import `os` on line 1\n"
    "## Ruff output\nAll checks passed!\n"
    "## Recommendations\n- Remove unused import\n"
)
GIT_LOG_OUTPUT = (
    "a1b2c3d feat: add OpenCode deployment manifests\n"
    "e4f5g6h docs: update README with MLflow setup\n"
    "i7j8k9l fix: correct volume mount path for skills"
)
GIT_DIFF_OUTPUT = (
    "diff --git a/deployment/opencode.yaml b/deployment/opencode.yaml\n"
    "--- /dev/null\n+++ b/deployment/opencode.yaml\n"
    "@@ -0,0 +1,45 @@\n+apiVersion: apps/v1\n+kind: Deployment\n"
    "+metadata:\n+  name: opencode-web\n"
)
GIT_STAT_OUTPUT = (
    " deployment/opencode.yaml | 45 +++++++++++++++\n"
    " docs/README.md          |  8 +++\n"
    " 2 files changed, 53 insertions(+)"
)
PR_SUMMARY = (
    "# PR #178: Add OpenCode deployment manifests\n"
    "## Summary\nAdds Kubernetes manifests for deploying OpenCode on OpenShift.\n"
    "## Changed files\n- deployment/opencode.yaml (new)\n- docs/README.md\n"
    "## Risk assessment\nLow — new files only, no existing code modified.\n"
    "## Test plan\n- Deploy to staging namespace\n"
)


# ── Trace 1: python-file-review — realistic clean run ───────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_clean(prompt: str):
    """Mirrors a real python-file-review execution: read, ruff, write report, verify."""
    mlflow.update_current_trace(
        tags={"skill": "python-file-review", "scenario": "clean"}
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": REVIEW_FILE})
        s.set_outputs({"content": SAMPLE_PYTHON_SOURCE})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": f"ruff check {REVIEW_FILE} --output-format=text"})
        s.set_outputs({"stdout": RUFF_OUTPUT_CLEAN, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": REPORT_FILE, "content": REVIEW_REPORT})
        s.set_outputs({"bytes_written": len(REVIEW_REPORT)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": REPORT_FILE})
        s.set_outputs({"content": REVIEW_REPORT[:200]})
    return (
        "Code review complete for data_pipeline.py. "
        "Report written to reviews/data_pipeline-review.md. "
        "Found 1 medium-severity issue: unused import `os`."
    )


# ── Trace 2: pr-summarizer — matches real cluster behavior ──────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_real(prompt: str):
    """Mirrors the real pr-summarizer behavior: writes summary WITHOUT read-back."""
    mlflow.update_current_trace(
        tags={"skill": "pr-summarizer", "scenario": "real-behavior"}
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git log main..pr/178 --oneline"})
        s.set_outputs({"stdout": GIT_LOG_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    return (
        "PR #178 summary written to pr-summaries/pr-178-summary.md. "
        "3 commits, 2 files changed. Low risk — new files only."
    )


print("Creating skill traces...")
review_clean("/skill python-file-review " + REVIEW_FILE)
pr_summary_real("/skill pr-summarizer 178")
mlflow.flush_trace_async_logging()
print("Created 2 traces mirroring real skill executions")

In [ ]:
from scorers import create_opencode_scorers

custom = create_opencode_scorers(
    judge_model=JUDGE_MODEL,
    groundedness_model=JUDGE_MODEL,
)

eval_traces = search_skill_traces()
print(f"Loaded {len(eval_traces)} skill traces for evaluation")

## 3. Evaluate Skill Traces

Run built-in and custom scorers against both skill traces using the same two-tier strategy as the [LangGraph evaluation notebook](../langgraph-end-to-end-example/langgraph_agent_evaluation.ipynb).

| Tier | Scorer | Type | What it checks |
|---|---|---|---|
| 1 | `DetectPII` | Built-in | PII entities in response (requires `guardrails-ai`) |
| 1 | `pii_check` | Custom | PII via regex patterns (no extra dependencies) |
| 1 | `tool_existence_check` | Custom | Called tools exist in known tool set |
| 1 | `repeated_action_loop` | Custom | Identical tool calls repeated 3+ times |
| 1 | `write_verification_check` | Custom | Every `tool_write` followed by `tool_read` of same path |
| 2 | `grounded_in_tools` | Custom | Response grounded in actual tool outputs |
| 2 | `semantic_loop_check` | Custom | Agent made progress vs stuck in a loop |
| 2 | `hallucination_check` | Custom | Response doesn't fabricate or contradict tool outputs |

> **Note:** MLflow's built-in `ToolCallEfficiency` scorer is not included — it requires tool definitions in `mlflow.chat.tools` span attributes, which OpenCode's trace format does not provide. Without them, the scorer always returns a false PASS. See the [Excessive Steps notebook](../failure-modes/03_excessive_steps/) to use it with agents that populate tool definitions.

### Tier 1: Deterministic checks

In [ ]:
tier1_scorers = [
    custom["pii_check"],
    custom["tool_existence_check"],
    custom["repeated_action_loop"],
    custom["write_verification_check"],
]

try:
    from mlflow.genai.scorers.guardrails import DetectPII

    tier1_scorers.append(DetectPII(pii_entities=["EMAIL_ADDRESS", "US_SSN"]))
    print("Including built-in DetectPII scorer")
except Exception:
    print("DetectPII not available (requires guardrails-ai) — using pii_check only")

with mlflow.start_run(run_name="tier1-deterministic"):
    tier1_results = mlflow.genai.evaluate(data=eval_traces, scorers=tier1_scorers)

print("\nTier 1 Results:")
for name, val in sorted(tier1_results.metrics.items()):
    print(f"  {name.replace('/mean', '')}: {float(val) * 100:.0f}% pass")

### Tier 2: LLM judges

Uses the configured judge model to evaluate nuanced aspects — tool efficiency, groundedness, semantic loops, hallucinations. Skipped when neither `OPENAI_API_BASE` nor `OPENAI_API_KEY` is set.

In [ ]:
if RUN_LLM_JUDGES:
    eval_traces = search_skill_traces()

    with mlflow.start_run(run_name="tier2-llm-judges"):
        tier2_results = mlflow.genai.evaluate(
            data=eval_traces,
            scorers=[
                custom["grounded_in_tools"],
                custom["semantic_loop_check"],
                custom["hallucination_check"],
            ],
        )

    print("Tier 2 Results:")
    for name, val in sorted(tier2_results.metrics.items()):
        print(f"  {name.replace('/mean', '')}: {float(val) * 100:.0f}% pass")
else:
    print(
        "Tier 2 skipped — set OPENAI_API_BASE or OPENAI_API_KEY to enable LLM judges."
    )

### Per-trace scorer results

In [ ]:
def _is_pass(val) -> bool:
    s = str(val).lower()
    if s in ("yes", "true"):
        return True
    if s in ("no", "false"):
        return False
    try:
        return float(val) >= 0.5
    except (ValueError, TypeError):
        return False


final_traces = search_skill_traces()

for i, t in enumerate(final_traces, 1):
    tags = t.info.tags or {}
    print(f"\n{'=' * 60}")
    print(f"Trace {i}: {tags.get('skill', '?')} — {tags.get('scenario', '?')}")
    print(f"{'=' * 60}")
    for a in t.info.assessments or []:
        if a.value is None:
            continue
        marker = "PASS" if _is_pass(a.value) else "FAIL"
        line = f"  {marker:4s} | {a.name}"
        if not _is_pass(a.value) and a.rationale:
            line += f"\n         {a.rationale[:150]}"
        print(line)

## 4. Interpret Results

The evaluation reveals a verification gap in `pr-summarizer`: the agent wrote its output but did not read it back to confirm the write succeeded. The `python-file-review` skill passed all scorers — it correctly performs read-back verification.

**What to expect:**
- **Tier 1 (deterministic)** results are stable across runs — `write_verification_check` will always FAIL for `pr-summarizer` and PASS for `python-file-review`.
- **Tier 2 (LLM judges)** results may vary between runs and across judge models. The verdicts depend on the specific model and endpoint used.

The screenshot below shows results from a specific cluster run (OpenCode 1.18.3, `gpt-oss-20b` via vLLM on OpenShift AI):

![MLflow assessments from cluster run](images/mlflow-assessments-right.png)

## 5. Strengthen Skill and Re-evaluate

The `write_verification_check` scorer found that `pr-summarizer` writes its output but skips the read-back step. The skill already instructed read-back (step 7), but the agent didn't comply. We strengthened the skill language to make verification mandatory (`MUST`, `required`), then re-evaluate to confirm the fix.

The cells below create two synthetic `pr-summarizer` traces — before (no read-back, matching the real failure) and after (with read-back) — and run `write_verification_check` on both.

> **Next step on cluster:** Remount the updated skill ConfigMap and re-run `/skill pr-summarizer 178` to capture a new real trace confirming the fix.

In [ ]:
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_no_verify(prompt: str):
    """Mirrors the real pr-summarizer failure: writes summary without read-back."""
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "pr-no-verification",
            "expected": "fail",
            "failure_mode": "verification_skipped",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    return "PR #178 summary written to pr-summaries/pr-178-summary.md."


@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_with_verify(prompt: str):
    """Expected passing pattern after skill strengthening: write then read-back."""
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "pr-fixed-verification",
            "expected": "pass",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": PR_SUMMARY_FILE})
        s.set_outputs({"content": PR_SUMMARY[:200]})
    return "PR #178 summary written and verified."


pr_summary_no_verify("/skill pr-summarizer 178")
pr_summary_with_verify("/skill pr-summarizer 178")
mlflow.flush_trace_async_logging()

before_trace = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'pr-no-verification'",
    return_type="list",
)
after_trace = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'pr-fixed-verification'",
    return_type="list",
)

with mlflow.start_run(run_name="before-after-pr-verification"):
    before_results = mlflow.genai.evaluate(
        data=before_trace, scorers=[custom["write_verification_check"]]
    )
    after_results = mlflow.genai.evaluate(
        data=after_trace, scorers=[custom["write_verification_check"]]
    )


def _verdict(metrics):
    val = next(
        (v for k, v in metrics.items() if k.startswith("write_verification_check")),
        None,
    )
    return "PASS" if val is not None and float(val) >= 0.5 else "FAIL"


print("Before (pr-summarizer without read-back — matches real trace failure):")
print(f"  {_verdict(before_results.metrics)} | write_verification_check")
print(f"  metrics: {before_results.metrics}\n")
print("After (pr-summarizer with read-back — expected passing pattern):")
print(f"  {_verdict(after_results.metrics)} | write_verification_check")
print(f"  metrics: {after_results.metrics}")

## 6. Summary

This notebook demonstrated the end-to-end evaluation workflow for OpenCode skills:

1. **Created traces** mirroring real skill executions observed on an OpenShift AI cluster
2. **Ran built-in + custom scorers** using a two-tier strategy (deterministic → LLM judges)
3. **Discovered a verification gap** — `pr-summarizer` writes its output without read-back confirmation
4. **Strengthened the skill** and confirmed the fix with before/after evaluation

The custom scorers in [`scorers.py`](scorers.py) extend the built-in MLflow scorers with checks specific to coding agents: write verification, hallucinated tool detection, repeated action loops, and PII detection (regex default, no extra dependencies). See the [failure-mode notebooks](../failure-modes/) for detailed coverage of each individual scorer.

### Trace export note

OpenCode 1.18.3's Go-based MLflow plugin creates trace metadata but does not upload span data as `traces.json` artifacts. Traces were reconstructed from OpenCode's local SQLite database using the Python SDK.